Harry Ferguson - CIS 361 - Summer 2026 - Assignment 9 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 904.32it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [37]:
train_df = pd.read_csv('spamham/train.csv')  
print(train_df['label'].value_counts())
print('Train data before conversion:\n',train_df.head(), '\n') 
train_df['label'] = train_df['label'].map({'spam': 1, 'ham': 0}) 
print('Train data after conversion:\n',train_df.head()) 

label
spam    2013
ham     1987
Name: count, dtype: int64
Train data before conversion:
   label                                               text
0   ham  Don't forget to bring the documents for tomorr...
1   ham            Let's catch up for coffee this weekend.
2   ham   Please find the attached report for your review.
3  spam  Get rich quick with this amazing investment op...
4   ham  Hi John, can we reschedule our meeting to next... 

Train data after conversion:
    label                                               text
0      0  Don't forget to bring the documents for tomorr...
1      0            Let's catch up for coffee this weekend.
2      0   Please find the attached report for your review.
3      1  Get rich quick with this amazing investment op...
4      0  Hi John, can we reschedule our meeting to next...


In [38]:
test_df = pd.read_csv('spamham/test.csv')  
print(test_df['label'].value_counts())
print('Test data before conversion:\n',test_df.head(), '\n') 
test_df['label'] = test_df['label'].map({'spam': 1, 'ham': 0}) 
print('Test data after conversion:\n',test_df.head()) 

label
ham     513
spam    487
Name: count, dtype: int64
Test data before conversion:
   label                                               text
0   ham            Let's catch up for coffee this weekend.
1  spam  Get rich quick with this amazing investment op...
2   ham  Don't forget to bring the documents for tomorr...
3  spam  Get rich quick with this amazing investment op...
4   ham   Please find the attached report for your review. 

Test data after conversion:
    label                                               text
0      0            Let's catch up for coffee this weekend.
1      1  Get rich quick with this amazing investment op...
2      0  Don't forget to bring the documents for tomorr...
3      1  Get rich quick with this amazing investment op...
4      0   Please find the attached report for your review.


In [19]:
# Tokenize the text data

def make_tensors(df):
    text = df['text'].tolist()
    encoded_dict = tokenizer(text, return_tensors="pt", padding='longest', truncation=True)
    labels = torch.tensor(df['label'].values)
    return encoded_dict.input_ids, encoded_dict.attention_mask, labels

train_input_ids, train_attention_masks, train_labels = make_tensors(train_df)
test_input_ids, test_attention_masks, test_labels = make_tensors(test_df)

# Convert to PyTorch DataLoader
BATCH_SIZE = 32
train_data = torch.utils.data.TensorDataset(train_input_ids, train_attention_masks, train_labels)
train_sampler = torch.utils.data.RandomSampler(train_data)
train_dataloader = torch.utils.data.DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

test_data = torch.utils.data.TensorDataset(test_input_ids, test_attention_masks, test_labels)
test_sampler = torch.utils.data.SequentialSampler(test_data)
test_dataloader = torch.utils.data.DataLoader(test_data, sampler=test_sampler, batch_size=BATCH_SIZE)

# Fine-tune the BERT model
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
EPOCHS = 3

In [20]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    avg_train_loss = total_loss / len(train_dataloader)

    print(f'Epoch {epoch + 1}/{EPOCHS}, Average Training Loss: {avg_train_loss}')

Epoch 1/3, Average Training Loss: 4.705906120580039e-05
Epoch 2/3, Average Training Loss: 4.561442388876458e-06
Epoch 3/3, Average Training Loss: 2.1604870626106275e-06


In [22]:
# Evaluate the model
model.eval()
preds = []
true_labels = []

for batch in test_dataloader:
    input_ids, attention_mask, labels = batch
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    preds.extend(logits.argmax(dim=1).cpu().numpy())
    true_labels.extend(labels.cpu().numpy())
accuracy = accuracy_score(true_labels, preds)
print(f'Accuracy: {accuracy}')

Accuracy: 1.0


In [39]:
def predict_spam(text, model, tokenizer):
    # Tokenize the text
    encoded_dict = tokenizer([text], return_tensors="pt", padding='longest', truncation=True) 

    # Perform inference
    with torch.no_grad():
        outputs = model(encoded_dict.input_ids, attention_mask=encoded_dict.attention_mask)

    logits = outputs.logits
    predicted_label = torch.argmax(logits, dim=1).item()

    # Map numerical label back to original label
    label_map = {1: 'spam', 0: 'ham'}
    return label_map[int(predicted_label)]

texts_to_predict = ("New Software Engineer at Max Tech", "Congratulations! You've won a free vacation. Click here to claim your prize!", "Carry budget-friendly tools in your wallet", 
                    "3 small habits mental performance coaches think could have the biggest impact", "sdfdfg 77hdadv", "Recent Canvas Notifications", "Warning! Your Cloud Storage Is Full",
                    "Avocado Sale: 25% OFF", "Your profile is popular - 11 search appearances", "Your purchase receipt", "Your daily usage for the past two weeks was 11 kWh.",
                     "My background check has cleared and I got my start date.")
for txt in texts_to_predict:
    predicted_label = predict_spam(txt, model, tokenizer)
    print(txt, ":", predicted_label)

New Software Engineer at Max Tech : ham
Congratulations! You've won a free vacation. Click here to claim your prize! : spam
Carry budget-friendly tools in your wallet : spam
3 small habits mental performance coaches think could have the biggest impact : ham
sdfdfg 77hdadv : spam
Recent Canvas Notifications : spam
Warning! Your Cloud Storage Is Full : spam
Avocado Sale: 25% OFF : spam
Your profile is popular - 11 search appearances : spam
Your purchase receipt : spam
Your daily usage for the past two weeks was 11 kWh. : ham
My background check has cleared and I got my start date. : spam


The training and test data set for this model comes from Kaggle: https://www.kaggle.com/datasets/abdulwadood11220/spam-or-real-email-dataset/. This dataset contains 5,000 sample emails labeled as either "spam" or "ham" (not spam) in 2 files, train.csv and test.csv. Although the website states that the training set has 4,000 emails and the test set has 1,000 emails, in fact the opposite is true. Since the training set is supposed to be the bigger file, I assume the creator of the data, Abdul Wadood, made a mistake in naming the files. For this model, I have correctly renamed the files.

Training the model appears to have been successful. Over the three epochs of training, the average training loss started very small at ~ .000047 and ended at `~ .00000216. In testing, the model earned an accuracy score of 1, meaning it made no mistakes in labeling the test emails. For this reason, I did not created a confusion matrix. But when it came time for the model to make predictions, the model made several mistakes. I tested the model with 12 sentences, 10 of which came from emails I've received (both ham and spam), and 2 made up phrases (Congratulations! You've won ... and sdfdfg ...). Although it correctly identified all of the spam emails, 4 ham emails (Carry budget-friendly tools..., Recent Canvas Notifications, Your profile is popular ..., and Your purchase receipt) were incorrectly labeled as spam. Seeing that the model earned a perfect  accuracy score on the test data, it is possible that the model is overtrained. Another possibilty is that the training set requires more emails, including emails with text similar to the 4 incorrectly labeled ham emails.